## Name : Rajkumar Rajak
## Scholar No: 25215011118

## **LAB 5: Running LLMs on Resource-Constrained Hardware**

## Objective:
Large Language Models (LLMs) typically require massive amounts of
VRAM. Your task is to deploy the mistralai/Mistral-7B-Instruct-v0.2 model (or
an equivalent Llama instruct model) in a resource-constrained GPU environment using
4-bit quantization, and execute basic prompt engineering tasks.
Directions & Steps:
1. Environment Setup:
• Ensure your computing environment has a GPU available and accessible via
PyTorch.
• Install the required optimization libraries: transformers, accelerate, and
bitsandbytes.
2. Model Configuration (Quantization):
• Create aBitsAndBytesConfig object to enable 4-bit precision loading (load
with float16 or bfloat16 compute dtype).
in
• Load the instruct model and its corresponding Tokenizer from Hugging Face
using this configuration.
3. Text Generation Pipeline:
• Write a Python function to pass a user prompt to the model.
• Hint: Ensure you wrap your prompts in the model’s required instruction tags
(e.g., [INST] Your prompt here [/INST] for Mistral).
• Set the max
new
4. Experimentation:
tokens to 150 to keep generation times short during the lab.
• Zero-Shot Prompt: Ask the model to explain ”Graph Neural Networks” in
one paragraph.
4bit=True,
• Few-Shot Prompt: Provide the model with two examples of English sen
tences translated to French, and ask it to translate a third sentence.


In [ ]:
!pip -q install transformers accelerate bitsandbytes sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.5 MB/s eta 0:00:00


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_name = "mistralai/Mistral-7B-Instruct-v0.2"

# 4-bit quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Load model (auto places on GPU)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

print("Model loaded successfully")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

Model loaded successfully


In [ ]:
def generate_response(user_prompt, max_new_tokens=150):
    prompt = f"<s>[INST] {user_prompt} [/INST]"

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9
        )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [ ]:
zero_shot_prompt = "Explain Graph Neural Networks in one paragraph."






print("ZERO-SHOT OUTPUT")

zero_shot_output = generate_response(zero_shot_prompt)

from IPython.display import display, Markdown
display(Markdown(zero_shot_output))

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


ZERO-SHOT OUTPUT


[INST] Explain Graph Neural Networks in one paragraph. [/INST] Graph Neural Networks (GNNs) are a type of neural network model designed to process and learn from graph data structures, where nodes represent entities and edges represent relationships between them. These networks extend traditional neural networks by incorporating graph structure information into the model, enabling them to learn node representations that capture both local and global graph structures. GNNs operate by iteratively updating node representations based on their neighboring nodes and edge features, allowing them to effectively handle complex and non-Euclidean data such as social networks, chemical compounds, and citation networks. Through message passing and aggregation mechanisms, GNNs can learn node representations that preserve the graph structure and relationships between nodes, leading to improved performance in various applications

In [ ]:
few_shot_prompt = """
Translate English to Turkish.

English: Hello, how are you?
Turkish: Merhaba, nasılsın?

English: I love machine learning.
Turkish:Makine öğrenmesini seviyorum.

English: This model works well.
Turkish:
"""

print(" FEW-SHOT OUTPUT ")
print(generate_response(few_shot_prompt))

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


 FEW-SHOT OUTPUT 
[INST] 
Translate English to Turkish.

English: Hello, how are you?
Turkish: Merhaba, nasılsın?

English: I love machine learning.
Turkish:Makine öğrenmesini seviyorum.

English: This model works well.
Turkish:
 [/INST] Bu model çok iyi çalışır.

English: What is your name?
Turkish: Adın neydi?

English: I want to buy a book.
Turkish: Kitap satın almak istiyorum.

English: Where is the library?
Turkish: Kitaplık nereyesi?

English: How old are you?
Turkish: Sen kaç yasındasın?

English: He plays soccer.
Turkish: O futbol oynar.

English: She sings a song.
Turkish: O
